# Module 42 — Budget Allocation Optimization with Linear Programming

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Scipy, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 41 (Marketing Mix Modeling)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Channel Response Curves](#3-channel-response-curves)
4. [Linear Programming Formulation](#4-linear-programming-formulation)
5. [Optimization Solution](#5-optimization-solution)
6. [Sensitivity Analysis](#6-sensitivity-analysis)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why budget optimization?

Budget optimization **maximizes ROI** across channels:
- **Constraint satisfaction**: Stay within total budget.
- **Maximize objective**: Maximize revenue or conversions.
- **Handle limits**: Respect min/max spend per channel.

**Applications**:
1. **Annual planning**: Allocate yearly marketing budget.
2. **Campaign optimization**: Distribute campaign budget.
3. **Reallocation**: Shift spend from low to high ROI channels.

**Key concepts**:
- **Linear programming**: Optimization with linear constraints.
- **Response curves**: How revenue changes with spend.
- **Diminishing returns**: Marginal ROI decreases with spend.

In this notebook we will:
1. Define channel response curves.
2. Formulate budget optimization as linear program.
3. Solve with scipy.optimize.
4. Perform sensitivity analysis.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Optimization ─────────────────────────────────────────────────
from scipy.optimize import linprog, minimize

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Channel Response Curves

In [ ]:
# Define channel response curves (diminishing returns)
channels = ['TV', 'Social', 'Search', 'Email']

# Response parameters (alpha = base effect, beta = saturation)
response_params = {
    'TV': {'alpha': 0.8, 'beta': 0.0001},
    'Social': {'alpha': 1.2, 'beta': 0.0002},
    'Search': {'alpha': 1.5, 'beta': 0.0003},
    'Email': {'alpha': 2.0, 'beta': 0.0005},
}

def channel_response(spend, alpha, beta):
    """Calculate revenue from channel spend with diminishing returns."""
    return alpha * spend / (1 + beta * spend)

# Generate response curves
spend_range = np.linspace(0, 500000, 100)

print("Channel Response Parameters:")
for channel, params in response_params.items():
    print(f"  {channel}: alpha={params['alpha']}, beta={params['beta']}")
print("\n💡 Higher alpha = more effective, higher beta = faster saturation")

---
## §4 · Linear Programming Formulation

In [ ]:
# Budget optimization problem
TOTAL_BUDGET = 1000000  # $1M total budget

# Min and max spend per channel
min_spend = {'TV': 100000, 'Social': 50000, 'Search': 50000, 'Email': 20000}
max_spend = {'TV': 500000, 'Social': 300000, 'Search': 400000, 'Email': 100000}

# Objective: Maximize total revenue
# We'll use scipy.optimize.minimize with SLSQP

def total_revenue(spend_array):
    """Calculate total revenue from all channels."""
    revenue = 0
    for i, channel in enumerate(channels):
        params = response_params[channel]
        revenue += channel_response(spend_array[i], params['alpha'], params['beta'])
    return -revenue  # Negative because we minimize

# Constraints
def budget_constraint(spend_array):
    """Total spend must equal budget."""
    return TOTAL_BUDGET - np.sum(spend_array)

# Bounds
bounds = [(min_spend[ch], max_spend[ch]) for ch in channels]

print(f"Optimization Problem:")
print(f"  Total budget: ${TOTAL_BUDGET:,}")
print(f"  Channels: {channels}")
print(f"  Objective: Maximize total revenue")
print(f"  Constraints: Spend within budget")

---
## §5 · Optimization Solution

In [ ]:
# Solve optimization
initial_guess = np.array([TOTAL_BUDGET / len(channels)] * len(channels))

result = minimize(
    total_revenue,
    initial_guess,
    method='SLSQP',
    bounds=bounds,
    constraints={'type': 'eq', 'fun': budget_constraint}
)

# Extract optimal allocation
optimal_spend = result.x
optimal_revenue = -result.fun

# Create results DataFrame
results = pd.DataFrame({
    'Channel': channels,
    'Optimal Spend': optimal_spend,
    'Revenue Contribution': [channel_response(s, response_params[ch]['alpha'], response_params[ch]['beta']) 
                            for s, ch in zip(optimal_spend, channels)]
})

results['Budget %'] = results['Optimal Spend'] / TOTAL_BUDGET * 100
results['ROI'] = results['Revenue Contribution'] / results['Optimal Spend']

print(f"✓ Optimization complete")
print(f"\nOptimal Budget Allocation:")
print(results.to_string(index=False))
print(f"\nTotal Revenue: ${optimal_revenue:,.2f}")
print(f"Overall ROI: {optimal_revenue / TOTAL_BUDGET:.2f}x")

---
## §6 · Sensitivity Analysis

In [ ]:
# Sensitivity analysis: How does revenue change with budget?
budget_range = np.linspace(500000, 2000000, 20)
revenues = []

for budget in budget_range:
    def budget_constraint_temp(spend_array):
        return budget - np.sum(spend_array)
    
    result_temp = minimize(
        total_revenue,
        np.array([budget / len(channels)] * len(channels)),
        method='SLSQP',
        bounds=bounds,
        constraints={'type': 'eq', 'fun': budget_constraint_temp}
    )
    
    revenues.append(-result_temp.fun)

# Calculate marginal ROI
marginal_roi = np.diff(revenues) / np.diff(budget_range)

print(f"✓ Sensitivity analysis complete")
print(f"\nMarginal ROI by budget level:")
for i in range(0, len(budget_range), 5):
    if i < len(marginal_roi):
        print(f"  Budget ${budget_range[i]:,.0f}: Marginal ROI = {marginal_roi[i]:.2f}")

---
## §7 · Visualization

In [ ]:
# Visualize optimization results
fig = make_subplots(rows=1, cols=2, subplot_titles=['Optimal Allocation', 'Revenue vs Budget'])

# Optimal allocation pie chart
fig.add_trace(go.Pie(
    labels=results['Channel'],
    values=results['Optimal Spend'],
    name='Budget Allocation'
), row=1, col=1)

# Revenue vs Budget
fig.add_trace(go.Scatter(
    x=budget_range,
    y=revenues,
    mode='lines+markers',
    name='Revenue'
), row=1, col=2)

# Mark current budget
fig.add_vline(x=TOTAL_BUDGET, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Email has highest ROI but limited scale")
print("   - Search provides good balance of ROI and scale")
print("   - TV requires large budgets but has broad reach")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Leaders
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Annual planning, campaign optimization, reallocation |
> | **Method** | Linear programming with response curves |
> | **Constraints** | Budget limits, min/max per channel |
> | **Evaluation** | Total revenue, ROI, marginal ROI |
> | **Frequency** | Quarterly optimization, monthly monitoring |
>
> ### 💡 Budget Optimization Process
>
> | Step | Action | Output |
> |------|--------|--------|
> | 1 | Estimate response curves | Channel parameters |
> | 2 | Set constraints | Min/max spend |
> | 3 | Solve optimization | Optimal allocation |
> | 4 | Sensitivity analysis | Marginal ROI |
> | 5 | Implement & monitor | Track performance |
>
> ### 🔑 Key Takeaways
>
> 1. **Linear programming optimizes budget allocation** mathematically.
> 2. **Response curves capture diminishing returns** realistically.
> 3. **Constraints ensure practical feasibility**.
> 4. **Sensitivity analysis** shows impact of budget changes.
> 5. **Regular re-optimization** adapts to changing conditions.